## KNN

In [ ]:
X1, X2, X3, X4, X5 = create_kfold_sets(data_train=data_train) #Model without feature selection
#X1, X2, X3, X4, X5 = create_kfold_sets(data_train=data_train_wrapper) #Model with feature selection

D = [X1, X2, X3, X4, X5]

In [ ]:
# Define the columns for the results DataFrame
columns = ['Model', 'N_neighbors',  'Distance', 'Accuracy',
           'Recall',
           'Specificity',
           'Precision',
           'F1']
df_results = pd.DataFrame(columns=columns)

model_name = 'KNN'

# List of neighbors to test
N_neighbors = [1, 3, 5, 11, 21, 41, 61]

# List of distance metrics to use
distance_metrics = ['euclidean', 'manhattan']

# Iterate over each number of neighbors and distance metric
for i in N_neighbors:
    for d in distance_metrics:
      df_results_fold = pd.DataFrame(columns=['Model', 'Accuracy', 'Recall', 'Specificity', 'Precision', 'F1'])

        # Perform cross-validation by iterating over each fold
        for j in range(5):

            # Select the test dataset for this iteration
            d_test = pd.concat([D[j]])
            y_test = d_test['label_binary']
            X_test = d_test.drop(columns=['label_binary', 'n_image', 'label_multi'])

            # Select the training datasets for this iteration
            d_train = pd.concat([D[k] for k in range(5) if k != j], ignore_index=True)
            y_train = d_train['label_binary']
            X_train = d_train.drop(columns=['label_binary', 'n_image', 'label_multi'])

            # Scale the data using StandardScaler
            scaler = StandardScaler()
            X_train = scaler.fit_transform(X_train)
            X_test = scaler.transform(X_test)

            # Initialize and fit the KNeighborsClassifier model
            model = KNeighborsClassifier(n_neighbors=i, metric=d)
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)

            results = evaluate_model(y_pred, y_test, model=model_name, labels=(1,0))
            df_results_fold = pd.concat([df_results_fold, results], ignore_index=True)
            print('\n')

        # Calculate mean metrics across all iterations
        recall_mean, specificity_mean, precision_mean, f1_mean, accuracy_mean = means_results(df_results_fold)

        # Create a dictionary to store the results for this model configuration
        result_i = {
            'Model': model_name,
            'Accuracy': accuracy_mean,
            'N_neighbors': i,
            'Distance': d,
            'Recall': recall_mean,
            'Specificity': specificity_mean,
            'Precision': precision_mean,
            'F1': f1_mean,
        }

        # Append the results to the DataFrame
        df_results = pd.concat([df_results, pd.DataFrame([result_i])], ignore_index=True)


In [ ]:
K = 61                        # Number of neighbors
DISTANCE_METRIC = 'manhattan' # Distance metric for KNN
LABELS = ('corrosion', 'no corrosion')  # Class labels
model_name = 'KNN'            # Name of the model

# ========================
# Data Preparation
# ========================
# Prepare training data
X_train = data_train.drop(columns=['label_binary', 'n_image', 'label_multi'])
y_train = data_train['label_binary']

# Prepare test data
X_test = data_test.drop(columns=['label_binary', 'n_image', 'label_multi'])
y_test = data_test['label_binary']

# ========================
# Feature Scaling
# ========================
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ========================
# Model Training
# ========================
model = KNeighborsClassifier(
    n_neighbors=K,
    metric=DISTANCE_METRIC
)
model.fit(X_train_scaled, y_train)

# ========================
# Model Evaluation
# ========================
# Time prediction process
start_time = time.time()
y_pred = model.predict(X_test_scaled)
execution_time = round(time.time() - start_time, 3)

recall, specificity, precision, f1, accuracy = evaluate_model(y_pred, y_test, model=model_name, labels=LABELS)

# ========================
# Results Storage
# ========================
results_columns = ['Model', 'N_neighbors', 'Distance', 'Accuracy',
                  'Recall', 'Specificity', 'Precision', 'F1', 'Time']

results_data = {
    'Model': 'KNN',
    'N_neighbors': K,
    'Distance': DISTANCE_METRIC,
    'Accuracy': accuracy,
    'Recall': recall,
    'Specificity': specificity,
    'Precision': precision,
    'F1': f1,
    'Time': execution_time
}

df_final_results = pd.DataFrame([results_data])
print('\nFinal Results DataFrame:')
print(df_final_results)